# Run Cycles for the head-to-head page

1. Download the Cycles input files from step 2 of the head-to-head page. Keep the zip as it is.
2. Click **Runtime > Run all**. When asked, choose that zip. If you used your own weather and soil files, choose those too.
3. Wait for the output zip to download, then drop it into step 3 of the head-to-head page.

This notebook downloads the public Cycles v1.4.4 release into this temporary Colab session and runs it there. Nothing of Cycles is stored in our repository.

In [ ]:
# Run Cycles v1.4.4 on the input files from the head-to-head page, then download one output zip.
# Click Runtime > Run all, choose the files when asked, and wait for the download at the end.
import glob, os, re, shutil, subprocess, sys, urllib.request, zipfile

CYCLES_URL = "https://github.com/PSUmodeling/Cycles/releases/download/v1.4.4/Cycles_debian_1.4.4.zip"
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).strip()

# 1. Cycles program (v1.4.4, Linux build, which is what Colab runs)
if not os.path.exists("Cycles"):
    local = sorted(glob.glob("Cycles_debian*.zip"))
    if not local:
        print("Downloading Cycles v1.4.4 ...")
        try:
            urllib.request.urlretrieve(CYCLES_URL, "Cycles_debian_1.4.4.zip")
        except Exception as e:
            sys.exit("Could not download Cycles (" + str(e) + "). Upload Cycles_debian_1.4.4.zip to this notebook's file panel and run this cell again.")
        local = ["Cycles_debian_1.4.4.zip"]
    with zipfile.ZipFile(local[0]) as z:
        z.extractall(".")
    os.chmod("Cycles", 0o755)
os.makedirs("input", exist_ok=True)
print("Cycles is ready.")

# 2. Your files: the zip from step 2 of the head-to-head page (plus your own .weather and .soil files, if you used those)
if IN_COLAB:
    print("Choose the zip you downloaded from the head-to-head page (and your own .weather/.soil files if you used them).")
    uploaded = list(files.upload().keys())
else:
    uploaded = [a for a in sys.argv[1:]]
for f in uploaded:
    if f.lower().endswith(".zip"):
        with zipfile.ZipFile(f) as z:
            for n in z.namelist():
                base = os.path.basename(n)
                if base and not n.endswith("/"):
                    with open(os.path.join("input", base), "wb") as out:
                        out.write(z.read(n))
    else:
        shutil.copy(f, os.path.join("input", os.path.basename(f)))

ctrls = [os.path.basename(c)[:-5] for c in glob.glob("input/*.ctrl")
         if os.path.basename(c)[:-5] not in ("ContinuousCorn", "CornSilageSoyWheat", "CornSoyWheatPasture")]
if not ctrls:
    sys.exit("No .ctrl file found. Upload the zip from step 2 of the head-to-head page.")
for NAME in ctrls:
    ctrl = open("input/" + NAME + ".ctrl").read()
    missing = [m for k in ("OPERATION_FILE", "SOIL_FILE", "WEATHER_FILE")
               for m in re.findall(k + r"\s+(\S+)", ctrl) if not os.path.exists("input/" + m)]
    if missing:
        sys.exit("Missing input file(s): " + ", ".join(missing) + ". Upload them too (for example your own .weather and .soil files) and run again.")

    # 3. Run Cycles
    print("Running Cycles for", NAME, "(1980-2016 takes about a minute) ...")
    code, text = sh("./Cycles -b " + NAME)
    if not os.path.exists("output/" + NAME + "/harvest.txt"):
        print(text[-1500:])
        sys.exit("Cycles did not produce output. The message above says why.")

    # 4. One output zip
    zname = NAME + "_output.zip"
    if os.path.exists(zname): os.remove(zname)
    with zipfile.ZipFile(zname, "w", zipfile.ZIP_DEFLATED) as z:
        for fn in sorted(os.listdir("output/" + NAME)):
            z.write(os.path.join("output", NAME, fn), fn)
    print("Done:", zname, "(" + str(len(os.listdir("output/" + NAME))) + " files)")
    if IN_COLAB:
        files.download(zname)
print("Now go back to the head-to-head page and drop the downloaded zip into step 3.")
